In [1]:
import math
import locale
import numpy as np
import mip
from collections import namedtuple
import glob
import pandas as pd
from IPython.display import display, Markdown

In [2]:
##### Open the main sheet #####

directory = "examples/Mariners"
globFilename = directory + "/Full Name*.xls*"
excelFiles = glob.glob(globFilename)
if len(excelFiles) < 1 or len(excelFiles) > 1:
    print(f"Can't find unique excel file: {globFilename}")
    exit()
mainSheet = pd.read_excel(excelFiles[0])

In [3]:
##### Find the constraint row #####
for constraintRow in range(len(mainSheet.Date)):
    if mainSheet.iat[constraintRow,2] == "At least":
        break

In [4]:
##### Establish an object for each day of the season
 
locale.setlocale(locale.LC_ALL, '')
Game = namedtuple('Game', ('weekday', 'month', 'day', 'gameDay', 'time', 'opponent', 'type', 'price', 'pairs', 'seats'))

##### Read the season schedule and store it as a list of games
 
openingDay = mainSheet.Date[0]
schedule = []
GamesInPlan = 0
PairsInPlan = 0
MaxPairsPerGame = 0
for weekday, date, time, opponent, type, price, pairs, seats in zip(mainSheet.Day, mainSheet.Date, mainSheet.Time, mainSheet.Opponent, mainSheet.Type, mainSheet.Price, mainSheet.GamePairs, mainSheet.Seats):
    if not isinstance(weekday, str) or weekday == "" or pd.isna(date):
        break
    schedule.append(Game(weekday, date.month, date.day, (date - openingDay).days, time.strftime("%I:%M %p"), opponent, type, price, pairs, seats))
    GamesInPlan += 1
    PairsInPlan += pairs
    MaxPairsPerGame = max(MaxPairsPerGame, pairs)


In [5]:
#########################################################
# Build a dictionary out of a list of games
#
#     buildCode == 0:    Create constraints for pair and quads
#     buildCode == 2:    Create constraint for pairs only
#     buildCode == 4:    Create constraint for quads only

def BuildDict(gameList, buildCode = 0):
    pairList = []
    for game in gameList:
        if buildCode != 4:
            pairList.append((game, 1.0))
        if buildCode != 2:
            pairList.append((game + GamesInPlan, 1.0))
    return dict(pairList)

# Define a class to contain constraints

class Constraint:
    def __init__(self, type, rhs, coefDict):
        self.type = type
        self.rhs = rhs
        self.coefDict = coefDict

# Define a class to contain each person's preferences

class SportsFan:
    def __init__(self, name, pairs, quads, ranking, extra):
        self.name = name
        self.pairs = pairs
        self.quads = quads

# Assign weights to games

        if len(ranking) != 0:
            self.ranking = ranking
        else:
            self.ranking = GamesInPlan * [GamesInPlan // 2]

# Adjust weights to favor highly ranked games

        self.useRanking = []
        midpoint = (GamesInPlan - 1) // 2
        for wgt in self.ranking:
            if wgt <= midpoint + 1:
                self.useRanking.append(math.sqrt(wgt - 1.0))
            else:
                self.useRanking.append(2.0 * math.sqrt(midpoint) - math.sqrt(2.0 * midpoint - wgt + 1))
        if len(self.useRanking) == GamesInPlan:
            self.useRanking += [2.0 * cost for cost in self.useRanking]
        else:
            for ix in range(GamesInPlan):
                self.useRanking[GamesInPlan + ix] *= 2

# Each person must attend correct number of games

        pairCon = Constraint('=', self.pairs, dict([(ix, 1.0) for ix in range(GamesInPlan)]))
        quadCon = Constraint('=', self.quads, dict([(ix, 1.0) for ix in range(GamesInPlan, 2 * GamesInPlan)]))

# Save all of the constraints for this person

        self.constraints = [pairCon, quadCon] + extra

##### This constraint handles the spacing of games

def Spacing(comparator, daysApart, pairsOrQuads = 0):
    firstGame = 0
    lastGame = 0
    constraintList = []
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].gameDay + daysApart + 1 > schedule[lastGame].gameDay:
            lastGame += 1
        constraintList.append(Constraint(comparator, 1, BuildDict(range(firstGame, lastGame), pairsOrQuads)))
        if lastGame == GamesInPlan:
            break
        while schedule[firstGame].gameDay + daysApart <= schedule[lastGame].gameDay:
            firstGame += 1
    return constraintList

##### Require or forbid games in various months

def Monthly(comparator, value, pairsOrQuads = 0):
    firstGame = 0
    lastGame = 0
    constraintList = []
    while True:
        while schedule[lastGame].month < 5:
            lastGame += 1
        while lastGame < GamesInPlan and schedule[firstGame].month == schedule[lastGame].month:
            lastGame += 1
        constraintList.append(Constraint(comparator, value, BuildDict(range(firstGame, lastGame), pairsOrQuads)))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraintList

##### Require or forbid games in different series

def Series(comparator, value, pairsOrQuads = 0):
    firstGame = 0
    lastGame = 0
    constraintList = []
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        constraintList.append(Constraint(comparator, value, BuildDict(range(firstGame, lastGame), pairsOrQuads)))
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    return constraintList

##### Require or forbid games for different opponents

def Opponents(comparator, value, pairsOrQuads = 0):
    firstGame = 0
    lastGame = 0
    opponentDict = {}
    while True:
        while lastGame < GamesInPlan and schedule[firstGame].opponent == schedule[lastGame].opponent:
            lastGame += 1
        opponentGames = opponentDict.get(schedule[firstGame].opponent, [])
        opponentGames += range(firstGame, lastGame)
        opponentDict[schedule[firstGame].opponent] = opponentGames
        if lastGame == GamesInPlan:
            break
        firstGame = lastGame
    constraintList = []
    for opponentGames in opponentDict.values():
        constraintList.append(Constraint(comparator, value, BuildDict(opponentGames, pairsOrQuads)))
    return constraintList


In [6]:
spreadSheetConstraints = namedtuple('Constraint', ('name', 'function', 'sense', 'pairsOrQuads'))
rowsToProcess = [spreadSheetConstraints("AtLeastPerMonth", Monthly, '>', 0),
                 spreadSheetConstraints("AtMostPerMonth", Monthly, '<', 0),
                 spreadSheetConstraints("AtLeastPerMonth(pairs)", Monthly, '>', 2),
                 spreadSheetConstraints("AtMostPerMonth(pairs)", Monthly, '<', 2),
                 spreadSheetConstraints("AtLeastPerMonth(quads)", Monthly, '>', 4),
                 spreadSheetConstraints("AtMostPerMonth(quads)", Monthly, '<', 4),
                 spreadSheetConstraints("AtLeastDaysBetween", Spacing, '>', 0),
                 spreadSheetConstraints("AtMostDaysBetween", Spacing, '<', 0),
                 spreadSheetConstraints("AtLeastDaysBetween(pairs)", Spacing, '>', 2),
                 spreadSheetConstraints("AtMostDaysBetween(pairs)", Spacing, '<', 2),
                 spreadSheetConstraints("AtLeastDaysBetween(quads)", Spacing, '>', 4),
                 spreadSheetConstraints("AtMostDaysBetween(quads)", Spacing, '<', 4),
                 spreadSheetConstraints("AtMostPerSeries", Series, '<', 0),
                 spreadSheetConstraints("AtMostPerOpponent", Opponents, '<', 0)]

In [7]:
##### Define the participants here #####

fans = []
totalPairs = 0
for fullName, nPairs, nQuads in zip(mainSheet.FullName, mainSheet.Pairs, mainSheet.Quads):
    if not isinstance(fullName, str) or fullName == "":
        break
    totalPairs += nPairs + 2 * nQuads
    globFilename = directory + "/" + fullName + "*.xls*"
    excelFiles = glob.glob(globFilename)
    if len(excelFiles) < 1 or len(excelFiles) > 1:
        print(f"Can't find unique excel file: {globFilename}")
        continue
    fanSheet = pd.read_excel(excelFiles[0])
    pairsRank = [np.int64(fanSheet.Pick[ix]) for ix in range(GamesInPlan)]
    if not isinstance(fanSheet.QuadPick[0], np.float64) and not math.isnan(fanSheet.QuadPick[0]):
        quadsRank = [np.int64(fanSheet.QuadPick[ix]) for ix in range(GamesInPlan)]
        pairsRank += quadsRank

    # Add fan constraints
    extraConstraints = []
    for row, rowConstraint in enumerate(rowsToProcess):
        if fanSheet.iat[constraintRow + row, 1]:
            constraintValue = fanSheet.iat[constraintRow + row, 3]
            extraConstraints += rowConstraint.function(rowConstraint.sense, constraintValue, rowConstraint.pairsOrQuads)
    fans.append(SportsFan(fullName, nPairs, nQuads, pairsRank, extraConstraints))

leftOver = PairsInPlan - totalPairs
if leftOver < 0:
    print("Too many games requested")
maxSparePairs = max((leftOver * GamesInPlan) // PairsInPlan, 1) # Maximum # of games that could be completely unassigned
while leftOver > 0:
    pairsRank = (GamesInPlan * [np.int64(1)])[:]
    nPairs = min(leftOver, maxSparePairs)
    fans.append(SportsFan(f"Spare Pair", nPairs, 0, pairsRank, []))
    leftOver -= nPairs


In [8]:
for gix, game in enumerate(schedule):
    picks = []
    for fan in fans:
        picks.append(int(fan.ranking[gix]))
    print(f"{picks} {game.weekday} {game.month}/{game.day} {game.opponent}{game.time} {game.type} {game.seats}")

[3, 63, 14, 1, 1] Thu 3/26 Guardians 07:10 PM A $90.00 (4)
[62, 30, 73, 1, 1] Fri 3/27 Guardians 06:45 PM B $69.00 (4)
[60, 36, 71, 1, 1] Sat 3/28 Guardians 06:40 PM B $69.00 (4)
[53, 28, 64, 1, 1] Sun 3/29 Guardians 04:20 PM B $69.00 (4)
[30, 29, 41, 1, 1] Mon 3/30 Yankees 06:40 PM D $45.00 (4)
[49, 9, 60, 1, 1] Tue 3/31 Yankees 06:40 PM D $45.00 (4)
[79, 31, 9, 1, 1] Wed 4/1 Yankees 01:10 PM D $45.00 (4)
[11, 42, 22, 1, 1] Fri 4/10 Astros 06:40 PM D $45.00 (4)
[26, 19, 37, 1, 1] Sat 4/11 Astros 06:40 PM D $45.00 (4)
[57, 43, 68, 1, 1] Sun 4/12 Astros 01:10 PM D $45.00 (4)
[73, 21, 3, 1, 1] Mon 4/13 Astros 01:10 PM B $69.00 (4)
[56, 3, 67, 1, 1] Fri 4/17 Rangers 06:40 PM B $69.00 (4)
[5, 77, 16, 1, 1] Sat 4/18 Rangers 04:15 PM B $69.00 (4)
[35, 35, 46, 1, 1] Sun 4/19 Rangers 01:10 PM B $69.00 (4)
[47, 65, 58, 1, 1] Mon 4/20 Athletics 06:40 PM B $69.00 (4)
[67, 46, 78, 1, 1] Tue 4/21 Athletics 06:40 PM B $69.00 (4)
[16, 68, 27, 1, 1] Wed 4/22 Athletics 01:10 PM D $45.00 (4)
[40, 73, 51

In [35]:
try:
    tixModel = mip.Model()
    tixVars = []
    for fan in fans:
        tixVars += [tixModel.add_var(name = fan.name + f"_pair_game_{ix}", var_type = mip.BINARY) for ix in range(GamesInPlan)]
        tixVars += [tixModel.add_var(name = fan.name + f"_quad_game_{ix}", var_type = mip.BINARY) for ix in range(GamesInPlan)]
    slackVars = [tixModel.add_var(name = f"slack_game_{ix}+", var_type = mip.CONTINUOUS, ub = 0.0) for ix in range(GamesInPlan)]
    slackVars += [tixModel.add_var(name = f"slack_game_{ix}-", var_type = mip.CONTINUOUS, ub = 0.0) for ix in range(GamesInPlan)]

    # All tickets must be allocated

    for ix in range(GamesInPlan):
        tixModel.add_constr(mip.xsum(tixVars[ix + 2 * iy * GamesInPlan] + 2.0 * tixVars[ix + GamesInPlan + 2 * iy * GamesInPlan] for iy in range(len(fans))) + slackVars[2 * ix] - slackVars[2 * ix + 1] == schedule[ix].pairs, name = f"Allocate_All_Tickets_game_{ix}")

    # Each fan must attend the correct number of games + satisfy all personal constraints

    for iy, fan in enumerate(fans):
        for ix, constraint in enumerate(fan.constraints):
            slackVars += [tixModel.add_var(name = f"slack_{fan.name}_constraint{ix}+", var_type = mip.CONTINUOUS, ub = 0.0)]
            slackVars += [tixModel.add_var(name = f"slack_{fan.name}_constraint{ix}-", var_type = mip.CONTINUOUS, ub = 0.0)]
            linFunc = mip.xsum(constraint.coefDict[iz] * tixVars[iz + iy * 2 * GamesInPlan] for iz in constraint.coefDict) + slackVars[-2] - slackVars[-1]
            if constraint.type == '=':
                tixModel.add_constr(linFunc == constraint.rhs, name = f"{fan.name}_constraint{ix}")
            if constraint.type == '<':
                tixModel.add_constr(linFunc <= constraint.rhs, name = f"{fan.name}_constraint{ix}")
            if constraint.type == '>':
                tixModel.add_constr(linFunc >= constraint.rhs, name = f"{fan.name}_constraint{ix}")

    # Establish the objective function

    costs = []
    for fan in fans:
        costs += fan.useRanking
    tixModel.objective = mip.xsum(costs[ix] * tixVars[ix] for ix in range(len(tixVars))) + mip.xsum(100000.0 * slackVars[ix] for ix in range(len(slackVars)))
except Exception as e:
    print(f"Mip setup exception: {e}")

In [36]:
try:
    status = tixModel.optimize()
except Exception as e:
    print(f"Mip optimize exception: {e}")

if status != mip.OptimizationStatus.OPTIMAL:
    print(f"No solution found: {status}")
    if status == mip.OptimizationStatus.INFEASIBLE:
        print("Infeasible solution found.  Relaxing constraints to find a solution with minimum slack.")
        tixRelax = tixModel.copy()
        for var in tixRelax.vars:
            if var.name.startswith("slack"):
                var.ub = 1.0
        try:
            status = tixRelax.optimize(max_seconds = 60)
        except Exception as e:
            print(f"Mip relax optimize exception: {e}")
        print(status)
        violatedConstraints = []
        for var in tixRelax.vars:
            if var.name.startswith("slack") and var.x > 0.0:
                violatedConstraints.append(var.name.removeprefix("slack_").removesuffix("+").removesuffix("-"))
        print("Violated constraints:  ", violatedConstraints)
        for constraint in tixRelax.constrs:
            if constraint.name in violatedConstraints:
                print(constraint)

No solution found: OptimizationStatus.INFEASIBLE
Infeasible solution found.  Relaxing constraints to find a solution with minimum slack.
OptimizationStatus.OPTIMAL
Violated constraints:   ['Tom Grandine_constraint2', 'Tom Grandine_constraint3', 'Tom Grandine_constraint4', 'Tom Grandine_constraint5', 'Tom Grandine_constraint6', 'Tom Grandine_constraint7']
Tom Grandine_constraint2: +1.0 Tom Grandine_pair_game_0 +1.0 Tom Grandine_pair_game_1 +1.0 Tom Grandine_pair_game_2
	 +1.0 Tom Grandine_pair_game_3 +1.0 Tom Grandine_pair_game_4 +1.0 Tom Grandine_pair_game_5
	 +1.0 Tom Grandine_pair_game_6 +1.0 Tom Grandine_pair_game_7 +1.0 Tom Grandine_pair_game_8
	 +1.0 Tom Grandine_pair_game_9 +1.0 Tom Grandine_pair_game_10 +1.0 Tom Grandine_pair_game_11
	 +1.0 Tom Grandine_pair_game_12 +1.0 Tom Grandine_pair_game_13 +1.0 Tom Grandine_pair_game_14
	 +1.0 Tom Grandine_pair_game_15 +1.0 Tom Grandine_pair_game_16 +1.0 Tom Grandine_quad_game_0
	 +1.0 Tom Grandine_quad_game_1 +1.0 Tom Grandine_quad_game_

In [12]:
picks = [[] for fan in fans]
for mix, mvar in enumerate(tixModel.vars):
    if mvar.x is not None and mvar.x > 0.5:
        fix = mix // (2 * GamesInPlan)
        if len(fans[fix].ranking) > GamesInPlan:
            gix = mix % (2 * GamesInPlan)
        else:
            gix = mix % GamesInPlan
        picks[fix].append(fans[fix].ranking[gix])
for fix, fan in enumerate(fans):
    picks[fix].sort()
    picks[fix] = [int(pick) for pick in picks[fix]]
    print(fans[fix].name, picks[fix])

Tom Grandine [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13]
Al Erisman [1, 2, 2, 3, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21]
Eric Brechner [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 13, 14, 17, 18, 19, 24, 27, 28, 30, 38, 49, 65]
Spare Pair [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Spare Pair [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [13]:
costs = {}
gameAllocationTable = "Day | Date | Time | Opponent | Type | Seats |"
for ix in range(MaxPairsPerGame):
    gameAllocationTable += f" Pair{ix+1} |"
gameAllocationTable += "\n| :-: | :-: | :-: | :-: | :-: | :-: |"  + " :-: |" * MaxPairsPerGame + "\n"
for gix, game in enumerate(schedule):
    gameAllocationTable += f"| {game.weekday} | {game.month}/{game.day} | {game.time} | {game.opponent} | {game.type} | {game.seats} |"
    for mix, mvar in enumerate(tixModel.vars):
        if mix % GamesInPlan == gix and mvar.x is not None and mvar.x > 0.5:
            fix = mix // (2 * GamesInPlan)
            if len(fans[fix].ranking) > GamesInPlan and mix % (2 * GamesInPlan) >= GamesInPlan:
                gix += GamesInPlan
            cost = costs.get(fans[fix].name, 0.0)
            costs[fans[fix].name] = cost + 2.0 * game.price
            gameAllocationTable += f" {fans[fix].name} ({fans[fix].ranking[gix]}) |"
            if mix % (2 * GamesInPlan) >= GamesInPlan:
                costs[fans[fix].name] += 2.0 * game.price
                gameAllocationTable += " |"
    gameAllocationTable += " ❌ |" * (MaxPairsPerGame - game.pairs) + "\n"
display(Markdown(gameAllocationTable))

Day | Date | Time | Opponent | Type | Seats | Pair1 | Pair2 |
| :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| Thu | 3/26 | 07:10 PM | Guardians  | A | $90.00 (4) | Tom Grandine (3) | Eric Brechner (14) |
| Fri | 3/27 | 06:45 PM | Guardians  | B | $69.00 (4) | Al Erisman (2) | |
| Sat | 3/28 | 06:40 PM | Guardians  | B | $69.00 (4) | Al Erisman (3) | |
| Sun | 3/29 | 04:20 PM | Guardians  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 3/30 | 06:40 PM | Yankees  | D | $45.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Tue | 3/31 | 06:40 PM | Yankees  | D | $45.00 (4) | Al Erisman (9) | Spare Pair (1) |
| Wed | 4/1 | 01:10 PM | Yankees  | D | $45.00 (4) | Eric Brechner (9) | Spare Pair (1) |
| Fri | 4/10 | 06:40 PM | Astros  | D | $45.00 (4) | Tom Grandine (11) | Spare Pair (1) |
| Sat | 4/11 | 06:40 PM | Astros  | D | $45.00 (4) | Al Erisman (19) | Spare Pair (1) |
| Sun | 4/12 | 01:10 PM | Astros  | D | $45.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 4/13 | 01:10 PM | Astros  | B | $69.00 (4) | Al Erisman (21) | Eric Brechner (3) |
| Fri | 4/17 | 06:40 PM | Rangers  | B | $69.00 (4) | Al Erisman (3) | Spare Pair (1) |
| Sat | 4/18 | 04:15 PM | Rangers  | B | $69.00 (4) | Tom Grandine (5) | |
| Sun | 4/19 | 01:10 PM | Rangers  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 4/20 | 06:40 PM | Athletics  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Tue | 4/21 | 06:40 PM | Athletics  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 4/22 | 01:10 PM | Athletics  | D | $45.00 (4) | Eric Brechner (27) | Spare Pair (1) |
| Fri | 5/1 | 06:45 PM | Royals  | D | $45.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sat | 5/2 | 06:40 PM | Royals  | A | $90.00 (4) | Eric Brechner (49) | Spare Pair (1) |
| Sun | 5/3 | 01:10 PM | Royals  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 5/4 | 06:40 PM | Braves  | A | $90.00 (4) | Eric Brechner (10) | Spare Pair (1) |
| Tue | 5/5 | 06:40 PM | Braves  | B | $69.00 (4) | Al Erisman (6) | Spare Pair (1) |
| Wed | 5/6 | 01:10 PM | Braves  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 5/15 | 06:40 PM | Padres  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sat | 5/16 | 04:15 PM | Padres  | D | $45.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 5/17 | 04:20 PM | Padres  | D | $45.00 (4) | Eric Brechner (65) | Spare Pair (1) |
| Mon | 5/18 | 06:40 PM | White Sox  | D | $45.00 (4) | Al Erisman (7) | Spare Pair (1) |
| Tue | 5/19 | 06:40 PM | White Sox  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 5/20 | 01:10 PM | White Sox  | B | $69.00 (4) | Al Erisman (8) | Eric Brechner (28) |
| Fri | 5/29 | 07:10 PM | D-backs  | B | $69.00 (4) | Tom Grandine (9) | Al Erisman (11) |
| Sat | 5/30 | 07:10 PM | D-backs  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 5/31 | 01:10 PM | D-backs  | C | $57.00 (4) | Al Erisman (18) | Eric Brechner (2) |
| Mon | 6/1 | 06:40 PM | Mets  | C | $57.00 (4) | Tom Grandine (1) | |
| Tue | 6/2 | 06:40 PM | Mets  | A | $90.00 (4) | Eric Brechner (30) | Spare Pair (1) |
| Wed | 6/3 | 12:40 PM | Mets  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Tue | 6/16 | 06:40 PM | Orioles  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 6/17 | 06:40 PM | Orioles  | B | $69.00 (4) | Al Erisman (5) | Eric Brechner (18) |
| Thu | 6/18 | 01:10 PM | Orioles  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 6/19 | 07:10 PM | Red Sox  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sat | 6/20 | 07:10 PM | Red Sox  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 6/21 | 01:10 PM | Red Sox  | C | $57.00 (4) | Tom Grandine (13) | Eric Brechner (24) |
| Mon | 6/29 | 06:40 PM | Angels  | C | $57.00 (4) | Al Erisman (15) | Spare Pair (1) |
| Tue | 6/30 | 06:40 PM | Angels  | B | $69.00 (4) | Al Erisman (1) | Spare Pair (1) |
| Thu | 7/2 | 06:40 PM | Angels  | A | $90.00 (4) | Eric Brechner (8) | Spare Pair (1) |
| Fri | 7/3 | 07:10 PM | Blue Jays  | A | $90.00 (4) | Eric Brechner (6) | Spare Pair (1) |
| Sat | 7/4 | 01:10 PM | Blue Jays  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 7/5 | 02:00 PM | Blue Jays  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 7/17 | 07:10 PM | Giants  | A | $90.00 (4) | Al Erisman (13) | Spare Pair (1) |
| Sat | 7/18 | 05:08 PM | Giants  | A | $90.00 (4) | Tom Grandine (8) | Eric Brechner (19) |
| Sun | 7/19 | 01:10 PM | Giants  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Mon | 7/20 | 06:40 PM | Reds  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Tue | 7/21 | 06:40 PM | Reds  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 7/22 | 12:40 PM | Reds  | B | $69.00 (4) | Eric Brechner (38) | Spare Pair (1) |
| Fri | 7/31 | 07:10 PM | Twins  | A | $90.00 (4) | Eric Brechner (7) | Spare Pair (1) |
| Sat | 8/1 | 01:10 PM | Twins  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 8/2 | 01:10 PM | Twins  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Tue | 8/4 | 06:40 PM | Tigers  | C | $57.00 (4) | Tom Grandine (12) | Spare Pair (1) |
| Wed | 8/5 | 06:40 PM | Tigers  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Thu | 8/6 | 01:10 PM | Tigers  | C | $57.00 (4) | Eric Brechner (4) | Spare Pair (1) |
| Fri | 8/7 | 07:10 PM | Rays  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sat | 8/8 | 06:50 PM | Rays  | A | $90.00 (4) | Tom Grandine (6) | Eric Brechner (17) |
| Sun | 8/9 | 01:10 PM | Rays  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 8/21 | 07:10 PM | Cubs  | B | $69.00 (4) | Al Erisman (10) | Spare Pair (1) |
| Sat | 8/22 | 06:40 PM | Cubs  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 8/23 | 01:10 PM | Cubs  | B | $69.00 (4) | Tom Grandine (2) | Eric Brechner (13) |
| Mon | 8/24 | 06:40 PM | Phillies  | B | $69.00 (4) | Tom Grandine (10) | Spare Pair (1) |
| Tue | 8/25 | 06:40 PM | Phillies  | B | $69.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 8/26 | 01:10 PM | Phillies  | B | $69.00 (4) | Eric Brechner (1) | Spare Pair (1) |
| Thu | 9/3 | 06:40 PM | Athletics  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 9/4 | 07:10 PM | Athletics  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sat | 9/5 | 06:40 PM | Athletics  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sun | 9/6 | 01:10 PM | Athletics  | C | $57.00 (4) | Al Erisman (4) | Spare Pair (1) |
| Tue | 9/8 | 06:40 PM | Rangers  | B | $69.00 (4) | Al Erisman (16) | Eric Brechner (5) |
| Wed | 9/9 | 06:40 PM | Rangers  | B | $69.00 (4) | Al Erisman (2) | Spare Pair (1) |
| Thu | 9/10 | 01:10 PM | Rangers  | B | $69.00 (4) | Al Erisman (17) | Spare Pair (1) |
| Tue | 9/22 | 06:40 PM | Astros  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Wed | 9/23 | 07:10 PM | Astros  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Thu | 9/24 | 06:40 PM | Angels  | C | $57.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Fri | 9/25 | 07:10 PM | Angels  | A | $90.00 (4) | Spare Pair (1) | Spare Pair (1) |
| Sat | 9/26 | 06:40 PM | Angels  | A | $90.00 (4) | Al Erisman (20) | Spare Pair (1) |
| Sun | 9/27 | 12:10 PM | Angels  | A | $90.00 (4) | Tom Grandine (4) | Al Erisman (12) |


In [14]:
amountOwedTable = "| | Amount owed | Paid |\n"
amountOwedTable += "| :- | -: | -: |\n"
for name, cost in sorted(costs.items()):
    amountOwedTable += f"| {name} | {locale.currency(cost, grouping=True)} | |\n"
display(Markdown(amountOwedTable))

| | Amount owed | Paid |
| :- | -: | -: |
| Al Erisman | $3,222.00 | |
| Eric Brechner | $3,198.00 | |
| Spare Pair | $13,860.00 | |
| Tom Grandine | $1,956.00 | |
